In [1]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage


from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = ""

True

In [2]:
import json

input_file = "../dataset/original_formatted/test.json"
output_file = "../dataset/ellipsis_recovered_formatted/test.json"

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [3]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are given a dialogue. Perform the following transformations:
1. Restore any omitted subjects, objects, or predicates. Example: change "고쳤어" → "[나는 창문을] 고쳤어".
2. Replace pronouns or vague references with specific nouns based on context. Example: change "이것" → "[사과]".
3. If the same speaker speaks consecutively, merge their utterances into one line so that the dialogue alternates between different speakers.
4. Do not transform unnecessary features.
Do not delete any sentence elements that do not fall under the above rules.

Mark only the modified parts with square brackets [ ].
Do not bracket unchanged words.
Keep the rest of the content exactly the same as the original except for the required modifications.



**Few-shot examples (mimic exactly this format)**

Example:
Input:
화자 2: 진짜 신의 한수
화자 1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아
화자 2: 글치 계속 해떴으면 몰랐겠지
화자 1: 그 때 물새는 거 알고 코킹작업해소 다행이다
화자 2: ㅇㅇ 안그랬으면 오늘처럼 비 많이 내리는 날 물바다됐을거야
화자 1: 요 아래 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물 안빠져서
화자 2: 새로 지은 곳인데도 그러네
화자 1: 부실공사지 뭐
화자 2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

Output:
화자 2: 진짜 [이번 상황은] 신의 한수
화자 1: [우리가] 이사하자마자 [장마철에] 비 많이 와서 [우리] 베란다 [창틀에서] 물 많이 새는 거 알았잖아
화자 2: [맞아, 그때] 계속 해떴으면 [물 새는 걸] 몰랐겠지
화자 1: [그 때 베란다에서] 물새는 거 알고 코킹작업해서 다행이다
화자 2: [응, 그때 코킹 안 했으면] 안 그랬으면 오늘처럼 비 많이 내리는 날 [베란다 안이] 물바다됐을 거야
화자 1: 요 아래 [아파트 단지 인근에서] 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 [인근 도로가] 땅 꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 [빗물이] 안 빠져서
화자 2: [여기는] 새로 지은 곳인데도 그러네
화자 1: [이건 완전히] 부실공사지 뭐
화자 2: 비 많이 올 때는 [그 공사 구간]으로 다니지 말아야겠다
화자 1: 응 조심해. 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

output only the transformed dialogue, nothing else. 
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual dialogue to Process>\n{actual_dialogue}")
])

chain = integrated_prompt | llm

In [4]:
import re
from copy import deepcopy

def strip_code_fences(text: str) -> str:
    if text is None:
        return ""
    # ```xxx\n ... \n``` 형태 제거
    fenced = re.compile(r"^\s*```(?:[a-zA-Z0-9_\-]+)?\s*\n(.*?)\n\s*```\s*$", re.DOTALL)
    m = fenced.match(text.strip())
    if m:
        return m.group(1).strip()
    return text.strip()

In [5]:
# ====== 변환 함수: 하나의 dialogue 문자열을 LLM으로 변환 ======
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from langchain_core.exceptions import OutputParserException

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=20),
    retry=retry_if_exception_type((TimeoutError, OutputParserException, Exception))
)
def transform_dialogue(dialogue_text: str) -> str:
    # prompt에 넣을 실제 대화
    inputs = {"actual_dialogue": dialogue_text}
    resp = chain.invoke(inputs)  # integrated_prompt | llm  (이미 위에서 정의됨)
    content = getattr(resp, "content", resp)  # 메시지 객체 or str 양쪽 대응
    return strip_code_fences(content)


In [6]:
# ====== 변환 함수: 하나의 dialogue 문자열을 LLM으로 변환 ======
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from langchain_core.exceptions import OutputParserException

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=20),
    retry=retry_if_exception_type((TimeoutError, OutputParserException, Exception))
)
def transform_dialogue(dialogue_text: str) -> str:
    # prompt에 넣을 실제 대화
    inputs = {"actual_dialogue": dialogue_text}
    resp = chain.invoke(inputs)  # integrated_prompt | llm  (이미 위에서 정의됨)
    content = getattr(resp, "content", resp)  # 메시지 객체 or str 양쪽 대응
    return strip_code_fences(content)


In [7]:
# ====== 데이터 로드(이미 위 셀에서 data 로드 완료 가정), 재시작 지원 ======
import os, json

# 기존 output이 있다면 이어붙이기(재시작용)
existing = {}
if os.path.exists(output_file):
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            prev = json.load(f)
            # id → item 매핑
            existing = {item.get("id"): item for item in prev if isinstance(item, dict) and "id" in item}
        print(f"[재시작] 기존 변환 {len(existing)}개 발견, 해당 항목은 건너뜁니다.")
    except Exception as e:
        print(f"[경고] 기존 출력 파싱 실패, 새로 생성합니다. 오류: {e}")
        existing = {}

# 처리 대상 id 집합
all_ids = [item.get("id") for item in data if isinstance(item, dict)]
print(f"총 입력 샘플 수: {len(all_ids)}")


총 입력 샘플 수: 605


In [8]:
# ====== 메인 루프: dialogue만 변환하고 동일 스키마 유지 ======
from tqdm import tqdm

converted_items = []

for item in tqdm(data, desc="Converting", ncols=100):
    if not isinstance(item, dict):
        continue

    item_id = item.get("id")
    if item_id in existing:
        # 기존 결과 사용(재시작 시)
        converted_items.append(existing[item_id])
        continue

    # 원본을 보존하면서 복사
    new_item = deepcopy(item)

    # 변환 대상: "dialogue" 필드
    original_dialogue = new_item.get("dialogue", "")
    transformed = transform_dialogue(original_dialogue)

    new_item["dialogue"] = transformed  # 대화만 교체, 나머지는 그대로 유지
    converted_items.append(new_item)

# ====== 저장 ======
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(converted_items, f, ensure_ascii=False, indent=2)

print(f"저장 완료 → {output_file} | 총 {len(converted_items)}개")


Converting: 100%|█████████████████████████████████████████████████| 605/605 [38:38<00:00,  3.83s/it]

저장 완료 → ../dataset/ellipsis_recovered_formatted/test.json | 총 605개
